# Chatbot 1

**Import Required Libraries**

These libraries support UI (Gradio), data handling (Pandas, NumPy), semantic search (FAISS), caching (JSON, OS), feedback logging (CSV), text embeddings (SentenceTransformer), and local LLM inference (Llama).

In [1]:
import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
c:\Users\brian\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\.libs\libopenblas64__v0.3.21-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


**Load FAISS Index and Chunked Text Data**

Loads the precomputed FAISS index and the chunked document dataset from CSV for similarity-based retrieval.

In [2]:
faiss_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\faiss_index_PaddleOCR.index"
csv_path = r"C:\Users\brian\OneDrive\Escritorio\Skills\Programming\Python\Project\extracted_text_PaddleOCR2\cleaned_text_PaddleOCR2\chunked_text_PaddleOCR.csv"
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)

**Load Models**

+ embedding_model: Transforms input queries into dense vector embeddings.
+ llm: Loads a quantized Mistral 7B model locally using llama-cpp.

In [3]:
# Load embedding model 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load quantized Mistral LLM 
mistral_path = r"C:\llama_cpp\llama-b5478-bin-win-cpu-x64\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
llm = Llama(model_path=mistral_path, n_ctx=2048)

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from C:\llama_cpp\llama-b5478-bin-win-cpu-x64\mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loa

**Set Up Cache and Feedback Logging**

Defines paths for saving cached Q&A pairs and storing user feedback.

In [4]:
# Cache and feedback log 
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

**Initialize Cache File**

Initializes an empty cache to avoid duplicate processing on repeated queries.

In [5]:
# Clear the cache JSON file
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump({}, f, indent=4, ensure_ascii=False)

**Utility Functions**

+ load_cache(): Loads the cache from file.

+ save_cache(): Saves updated Q&A pairs to cache.

+ save_feedback(): Appends feedback entries to a CSV file.

In [6]:
# === Utility Functions ===
def load_cache():
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)

def save_feedback(query, answer, source, feedback):
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    file_exists = os.path.isfile(FEEDBACK_FILE)
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data)

**Semantic Search: Top-K Retrieval**
+ Encodes the query into a vector and finds the top k most similar text chunks from FAISS.

+ Returns a subset of df_chunks.

In [7]:
def search_top_k(query, k=2):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding).astype("float32"), k)
    return df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]

**Prompt Builder**
Combines the retrieved text chunks and the question into a structured prompt format for the LLM.

In [8]:
def build_prompt(chunks, question):
    context = "\n".join(chunks["chunk_text"].tolist())
    prompt = (
        "You are a helpful assistant. Use the information in the context below to answer the user's question.\n\n"
        "### Context ###\n"
        f"{context}\n\n"
        "### Question ###\n"
        f"{question}\n\n"
        "### Answer ###\n"
    )
    return prompt

**Generate Answer with LLM**
+ Builds the prompt and sends it to the Mistral model to generate an answer.
+ Returns the model’s textual response.

In [9]:
def generate_answer(chunks, question):
    prompt = build_prompt(chunks, question)
    response = llm(prompt, max_tokens=200, stop=["</s>", "###"])
    return response["choices"][0]["text"].strip()

**Main Chatbot Function (Gradio)**
+ Checks if the answer is already in cache.

+ If not, performs retrieval and generation.

+ Updates and returns the conversation history.

In [10]:
# === Gradio Chat Function ===
cache = load_cache()


def chatbot_ui(query, history):
    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
        from_cache = True
    else:
        chunks = search_top_k(query)
        answer = generate_answer(chunks, query)
        source = chunks.iloc[0]["filename"]
        cache[query] = {"answer": answer, "source": source}
        save_cache(cache)
        from_cache = False

    display_answer = f"💬 **Answer:** {answer}\n📄 **Source:** {source}"
    history.append((query, display_answer))
    return history, query, answer, source

**Feedback Button Handler**

+ Logs the user’s feedback to a CSV file.

+ Displays a feedback confirmation in the chat.

In [11]:
def feedback_fn(query, answer, source, feedback, history):
    save_feedback(query, answer, source, feedback)
    return history + [("✅ Feedback received: " + feedback.upper(), "")]

# === Launch UI ===


**Launch the Gradio Interface**

+ Builds the interactive chatbot interface with:
    + Input box for questions
    + Chat history display
    + Buttons for submit and feedback
+ Launches the web UI using Gradio.

In [12]:
with gr.Blocks(title="Chatter - RAG Chatbot") as demo:
    gr.Markdown("## 💬 Chatter: RAG-based Chatbot with RLHF")
    chatbot = gr.Chatbot()
    query = gr.Textbox(label="Ask something...")
    answer_box = gr.Textbox(visible=False)
    source_box = gr.Textbox(visible=False)

    with gr.Row():
        btn_submit = gr.Button("Submit")
        btn_good = gr.Button("👍")
        btn_bad = gr.Button("👎")

    state_query = gr.State()
    state_answer = gr.State()
    state_source = gr.State()

    btn_submit.click(chatbot_ui, inputs=[query, chatbot], outputs=[chatbot, state_query, state_answer, state_source])
    btn_good.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="good"), chatbot], outputs=[chatbot])
    btn_bad.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="bad"), chatbot], outputs=[chatbot])

demo.launch()



C:\Users\brian\AppData\Local\Temp\ipykernel_4564\246883693.py:3: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
